In [3]:
import os
from glob import glob
import numpy as np    
import pandas as pd
import matplotlib.pyplot as plt
import numpy.linalg as la

def calX25(Vcmax,J,TPU,Rd,gm,T):
    c_O2 = 0.21           
    R = 8.314e-3    # [kJ K-1 mol-1]
    c_Vcmax = 26.355
    Ha_Vcmax = 65.33
    c_J = 17.71
    Ha_J = 43.9
    c_TPU = 21.46
    Ha_TPU = 53.1
    Hd_TPU = 201.8
    dS_TPU = 0.65
    c_Rd = 18.7145
    Ha_Rd = 46.39
    c_gm = 20.01
    Ha_gm = 49.6
    Hd_gm = 437.4
    dS_gm = 1.4
    Vcmax25 = Vcmax / (np.exp(c_Vcmax-(Ha_Vcmax/(R*T))))
    J25 = J / np.exp(c_J-(Ha_J/(R*T)))
    TPU25 = TPU / ((np.exp(c_TPU-(Ha_TPU/(R*T))) / (1+np.exp((dS_TPU*T-Hd_TPU)/(R*T)))))
    Rd25 = Rd / np.exp(c_Rd-(Ha_Rd/(R*T)))
    gm25 = gm / ((np.exp(c_gm-(Ha_gm/(R*T))) / (1+np.exp((dS_gm*T-Hd_gm)/(R*T)))))
    Tref = 298.15
    GammaS25 = np.exp(11.187 - 24.46 / (8.314e-3 * Tref))         
    return Vcmax25,J25,TPU25,Rd25,gm25,GammaS25

def SharkeyC3(Vcmax, J, TPU, Rd, Cc, T, P):
    
    c_O2 = 0.21          
    R = 8.314e-3    # [kJ K-1 mol-1] 
    O = P * c_O2    # [Pa]
    c_KC = 35.9774
    Ha_KC = 80.99
    c_KO = 12.3772
    Ha_KO = 23.72
    c_GammaS = 11.187
    Ha_GammaS = 24.46
    KC = np.exp(c_KC-Ha_KC/(R*T))    # [Pa]
    KO = np.exp(c_KO-Ha_KO/(R*T)) * 1000    # [Pa]
    GammaS = (np.exp(c_GammaS-Ha_GammaS/(R*T)))     # [Pa]
    K = KC*(1+O/KO)    # [Pa]
    
    Ac = Vcmax * (Cc-GammaS)/(Cc+K) - Rd
    Aj = J * (Cc-GammaS)/(4*Cc+8*GammaS) - Rd
    At = 3*TPU - Rd
    At = np.full(Cc.size,At)
    
    An = np.nanmin([Ac,Aj,At],0)
    
    return An,Ac,Aj,At
    
def SharkeyC3_(A, Rd, limit, Cc, T, P):  
    
    c_O2 = 0.21  

    R = 8.314e-3    # [kJ K-1 mol-1] 
    O = P * c_O2    # [Pa]
    
    c_KC = 35.9774
    Ha_KC = 80.99
    c_KO = 12.3772
    Ha_KO = 23.72
    c_GammaS = 11.187
    Ha_GammaS = 24.46
    KC = np.exp(c_KC-Ha_KC/(R*T))    # [Pa]
    KO = np.exp(c_KO-Ha_KO/(R*T)) * 1000    # [Pa]
    GammaS = (np.exp(c_GammaS-Ha_GammaS/(R*T)))  # [Pa]
    K = KC*(1+O/KO)    # [Pa] 
    
    X = (Cc-GammaS)/(Cc+K)
    Y = A + Rd
    X = X[limit==1].reshape(1,-1)
    Y = Y[limit==1].reshape(1,-1)
    Vcmax = np.dot(Y,la.pinv(X))
    Vcmax = Vcmax.item()
    
    X = (Cc-GammaS)/(4*Cc+8*GammaS)
    Y = A + Rd
    X = X[limit==2].reshape(1,-1)
    Y = Y[limit==2].reshape(1,-1)
    J = np.dot(Y,la.pinv(X))
    J = J.item()
    
    X = np.full(Cc.size,3)
    Y = A + Rd
    if (limit==3).sum() > 0:
        X = X[limit==3].reshape(1,-1)
        Y = Y[limit==3].reshape(1,-1)
        TPU = np.dot(Y,la.pinv(X))
        TPU = TPU.item()
    else:
        TPU = np.nan
    
    Ac = Vcmax * (Cc-GammaS)/(Cc+K) - Rd
    Aj = J * (Cc-GammaS)/(4*Cc+8*GammaS) - Rd
    At = 3*TPU - Rd
    At = np.full(limit.size,At)
    
    
    An = np.empty(limit.size)
    An[limit==1] = Ac[limit==1]
    An[limit==2] = Aj[limit==2]
    An[limit==3] = At[limit==3]
    
    return An,Ac,Aj,At,Vcmax,J,TPU,GammaS
    
def resi(p, x, y):
    limit = x[0]
    Ci = x[1]
    T = x[2]
    P = x[3]
    Rd = p[0]
    gm = p[1]
    Cc = Ci - y/gm
    An = SharkeyC3_(y, Rd, limit, Cc, T, P)[0]
    return y - An   

## Set your folders
names = ['raw20190627SY','raw20190709SY','raw20190728SY','raw20190803SY','raw20190810SY','raw20190818SY','raw20190823SY','raw20190907SY','raw20200615SY']

## Batch
for name in names:
    input = '../data/soybean2019/%s' % name
    output = '../data/soybean2019_result/sharkey_2007_%s' % name

    os.makedirs(output,exist_ok=True)
    files = glob('%s/*.xlsx' % input)
    files.sort()
    # result = open('%s/result.csv' % output,'w')
    # result.write('file,Vcmax25,J25,Rd25,J25/Vcmax25,R2,RMSE\n')
    file_names = []
    Vcmax25s = []
    J25s = []
    Vcmax_list = []
    J_list = []
    TPU_list = []
    Rd_list = []
    gm_list = []
    GammaS_list =[]
    R2_list =[]
    RMSE_list = [] 
    Ao_list = [] 
    Ann_list = [] 

    for file in files:
        print(file)
        
        ## Read raw data
        xlsx = pd.read_excel(file,skiprows=14)
        A = np.array(xlsx['A'][1:],np.float32)
        T = np.array(xlsx['Tleaf'][1:],np.float32) + 273.15    # [K]
        P = np.array(xlsx['Pa'][1:],np.float32) * 1000    # [Pa]
        #Ca = np.array(xlsx['Ca'][1:],np.float32)   # [ppm]
        Ci = np.array(xlsx['Ci'][1:],np.float32)*1e-6 * P    # [Pa]
   
    
        ## Set criteria to remove points
        msk = np.full(A.size,True)
        msk[(Ci > 20) & (Ci < 30)] = False
        A = np.array(A[msk])
        Ci = np.array(Ci[msk])
        T = T[msk].mean()
        P = P[msk].mean()    
        
        ## Sort from small Ci to large Ci
        idx = np.argsort(Ci)
        A = A[idx]
        Ci = Ci[idx]
        
        ## Set criteria for limiting factors. 1: Ac; 2: Aj; 3: At
        limit = np.full(A.size,2)    # Default is Aj
        limit[Ci<20] = 1    # This means Ci < 20 Pa are Ac
        limit[-1:] = 3    # This means the last one is At
    
        ## Rough estimate
        Rd0 = np.arange(0,10.01,0.1)
        gm0 = np.arange(0,30.01,0.1)
        gm0[0] = 1e-5
        RMSE = np.empty([Rd0.size,gm0.size])
        for i,Rd in enumerate(Rd0):
            for j,gm in enumerate(gm0):
                RMSE[i,j] = (resi([Rd,gm],[limit,Ci,T,P],A)**2).sum()
        Rd = Rd0[np.where(RMSE==np.min(RMSE))[0][0]]
        gm = gm0[np.where(RMSE==np.min(RMSE))[1][0]]
        Cc = Ci - A/gm
        An,Ac,Aj,At,Vcmax,J,TPU,GammaS = SharkeyC3_(A, Rd, limit, Cc, T, P)
        R2 = np.corrcoef(A,An)[0,1]**2
        RMSE = ((An-A)**2).sum()

        ## Accurate estimate
        Rd0 = np.arange(np.max([0,Rd-1]),np.min([10,Rd])+0.01,0.1)
        gm0 = np.arange(np.max([0,gm-1]),np.min([30,gm])+0.01,0.1)
        gm0[0] = 1e-5
        RMSE = np.empty([Rd0.size,gm0.size])
        for i,Rd in enumerate(Rd0):
            for j,gm in enumerate(gm0):
                RMSE[i,j] = (resi([Rd,gm],[limit,Ci,T,P],A)**2).sum()
        Rd = Rd0[np.where(RMSE==np.min(RMSE))[0][0]]
        gm = gm0[np.where(RMSE==np.min(RMSE))[1][0]]
        Cc = Ci - A/gm
        An,Ac,Aj,At,Vcmax,J,TPU,GammaS = SharkeyC3_(A, Rd, limit, Cc, T, P)
        R2 = np.corrcoef(A,An)[0,1]**2
        RMSE = np.sqrt(np.mean((A - An) ** 2))
        Ao = A.mean()
        Ann = An.mean()
        

        ## 25C
        Vcmax25,J25,TPU25,Rd25,gm25,GammaS25 = calX25(Vcmax,J,TPU,Rd,gm,T)

        file_names.append(file.split('\\')[-1]+',')
        Vcmax25s.append(Vcmax25) 
        J25s.append(J25) 
        Vcmax_list.append(Vcmax) 
        J_list.append(J) 
        Rd_list.append(Rd)
        gm_list.append(gm)
        TPU_list.append(TPU)
        GammaS_list.append(GammaS)
        R2_list.append(R2)
        RMSE_list.append(RMSE)
        Ao_list.append(Ao)
        Ann_list.append(Ann)
  

  
        ## Write
    df0 = pd.DataFrame(file_names, columns=['filenames'])
    df1 = pd.DataFrame(Vcmax25s, columns=['Vcmax25'])
    df2 = pd.DataFrame(J25s, columns=['J25'])
    df7 = pd.DataFrame(Vcmax_list, columns=['Vcmax'])
    df8 = pd.DataFrame(J_list, columns=['J'])
    df9 = pd.DataFrame(Rd_list, columns=['Rd'])
    df10 = pd.DataFrame(TPU_list, columns=['TPU'])
    df11 = pd.DataFrame(gm_list, columns=['gm'])
    df12 = pd.DataFrame(GammaS_list, columns=['GammaS'])
    df13 = pd.DataFrame(R2_list, columns=['R2'])
    df14 = pd.DataFrame(RMSE_list, columns=['RMSE'])
    df15 = pd.DataFrame(Ao_list, columns=['A'])
    df16 = pd.DataFrame(Ann_list, columns=['An'])
   

    df_combined = pd.concat([df0,df1,df2,df7,df8,df9,df10,df11,df12,df13,df14,df15,df16], axis=1)

    filename = output + '/result.csv'
    # print(filename)
    df_combined.to_csv(filename,index=False)    




    

D:/博士学习资料/根据Vcmax25计算A/大豆01/raw20190627SY\2019-06-27-ACi-top2.xlsx
D:/博士学习资料/根据Vcmax25计算A/大豆01/raw20190627SY\2019-06-27-ACi-top3.xlsx
D:/博士学习资料/根据Vcmax25计算A/大豆01/raw20190627SY\2019-06-27-ACi-top4.xlsx
D:/博士学习资料/根据Vcmax25计算A/大豆01/raw20190627SY\2019-06-27-ACi-top5.xlsx
